# CAFREC — Dataset Builder: KuaiRand → RecBole Atomic Files

**Project:** CAFREC — Context-Adaptive Fusion for Long- and Short-Term Video Recommendation
**Task:** Block 1 dataset build (feeds T1.2 baseline reproduction — RON-10 / RON-11)
**Author:** R. Francis

---

## What this notebook does

The T1.1 preparation notebook profiles **KuaiRand-Pure** and writes parquet feature
artifacts. This notebook is the complementary build step: it turns the **raw
KuaiRand CSVs into RecBole `.inter` atomic files** for **three tiers**, so the
`cafrec_harness` baselines (SASRec, HGN) can train at three scales:

| Tier | KuaiRand variant | Raw size | RecBole dataset name |
|------|------------------|----------|----------------------|
| **light**  | KuaiRand-Pure | ~184 MB  | `kuairand_pure` |
| **medium** | KuaiRand-1K   | ~4.3 GB  | `kuairand_1k`   |
| **heavy**  | KuaiRand-27K  | ~46 GB   | `kuairand_27k`  |

**Assumptions (per build decision):**

* The raw files are **already downloaded and extracted** — no download logic here.
  Expected layout, per tier:

  ```
  data/KuaiRand-Pure/data/log_standard_4_08_to_4_21_pure.csv   (+ 2 more logs)
  data/KuaiRand-1K/data/log_standard_4_08_to_4_21_1k.csv       (+ 2 more logs)
  data/KuaiRand-27K/data/log_standard_4_08_to_4_21_27k.csv     (+ 2 more logs)
  ```

* Output is **`.inter` only** — no parquet. The `.inter` schema
  (`user_id:token`, `item_id:token`, `timestamp:float`) is exactly what
  `configs/base.yaml` loads, and its `eval_args` already specify leave-one-out
  with 99 uniform negatives (`uni100`), so no further harness changes are needed.


## 1  Imports & configuration

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

RANDOM_SEED = 403092          # project-standard seed
np.random.seed(RANDOM_SEED)

# --- paths ------------------------------------------------------------------
# Run from the project root (the folder containing data/ and cafrec_harness/).
DATA_DIR    = Path("data")            # where raw KuaiRand-* folders live
OUTPUT_ROOT = Path("data/recbole")    # RecBole reads <OUTPUT_ROOT>/<dataset>/<dataset>.inter

# tier -> raw variant folder, in-file suffix, and the RecBole dataset name emitted
VERSIONS = {
    "light":  {"variant": "KuaiRand-Pure", "suffix": "pure", "dataset": "kuairand_pure"},
    "medium": {"variant": "KuaiRand-1K",   "suffix": "1k",   "dataset": "kuairand_1k"},
    "heavy":  {"variant": "KuaiRand-27K",  "suffix": "27k",  "dataset": "kuairand_27k"},
}

# --- what counts as an interaction -----------------------------------------
# ORGANIC_ONLY: keep only standard-recommendation rows (is_rand == 0). Random-
#   policy rows are researcher interventions (T1.1 decision record) and are
#   never used as a training/eval target.
ORGANIC_ONLY = True

# POSITIVE_RULE: which rows enter the interaction sequence.
#   "click"          -> is_click == 1   (KuaiRand valid-play/click signal; default)
#   "effective_view" -> play_time_ms / duration_ms >= EFFECTIVE_VIEW_THRESH
#   "all"            -> every impression (noisiest; keeps skipped videos)
POSITIVE_RULE         = "click"
EFFECTIVE_VIEW_THRESH = 0.5

# --- filtering --------------------------------------------------------------
# Leave-one-out needs >= 3 interactions/user (train + val + test). MIN_USER_INTER
# raises that floor for training stability. Item filtering is OFF by default so
# the full catalogue (and the Coverage denominator) is preserved.
MIN_USER_INTER = 5
MIN_ITEM_INTER = 0
K_CORE_ITERATE = False     # True -> iterate user/item filters to convergence (true k-core)

# --- IO ---------------------------------------------------------------------
CHUNKSIZE = 2_000_000      # stream large logs (27K) in chunks; filter on read
USE_COLS  = ["user_id", "video_id", "time_ms", "is_rand", "is_click",
             "play_time_ms", "duration_ms"]
DTYPES    = {"user_id": "int32", "video_id": "int32", "time_ms": "int64",
             "is_rand": "int8", "is_click": "int8",
             "play_time_ms": "int64", "duration_ms": "int64"}

print("Config loaded.")
print(f"  tiers          : {list(VERSIONS)}")
print(f"  positive rule  : {POSITIVE_RULE}  |  organic only: {ORGANIC_ONLY}")
print(f"  min user inter : {MIN_USER_INTER}  |  min item inter: {MIN_ITEM_INTER}")
print(f"  output root    : {OUTPUT_ROOT.resolve()}")


## 2  Helpers — filter-on-read, positive rule, k-core

In [ ]:
def _passes_positive(df):
    # Boolean mask implementing POSITIVE_RULE for a dataframe chunk.
    if POSITIVE_RULE == "click":
        return df["is_click"] == 1
    if POSITIVE_RULE == "effective_view":
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(df["duration_ms"] > 0,
                             df["play_time_ms"] / df["duration_ms"], 0.0)
        return ratio >= EFFECTIVE_VIEW_THRESH
    if POSITIVE_RULE == "all":
        return np.ones(len(df), dtype=bool)
    raise ValueError(f"unknown POSITIVE_RULE: {POSITIVE_RULE}")


def load_and_filter(path):
    # Stream one raw log in chunks, applying organic + positive filters on read
    # so peak memory stays bounded even for KuaiRand-27K. Keeps only the three
    # columns an atomic file needs.
    keep, kept = ["user_id", "video_id", "time_ms"], []
    for chunk in pd.read_csv(path, usecols=USE_COLS, dtype=DTYPES, chunksize=CHUNKSIZE):
        if ORGANIC_ONLY:
            chunk = chunk[chunk["is_rand"] == 0]
        chunk = chunk[_passes_positive(chunk)]
        kept.append(chunk[keep])
    return pd.concat(kept, ignore_index=True) if kept else pd.DataFrame(columns=keep)


def k_core_filter(df, min_user, min_item, iterate):
    # Drop users/items below thresholds. Single pass unless iterate=True, in
    # which case repeat until row count stabilises (proper k-core).
    while True:
        n0 = len(df)
        if min_user > 0:
            uc = df["user_id"].value_counts()
            df = df[df["user_id"].isin(uc[uc >= min_user].index)]
        if min_item > 0:
            ic = df["video_id"].value_counts()
            df = df[df["video_id"].isin(ic[ic >= min_item].index)]
        if not iterate or len(df) == n0:
            break
    return df.reset_index(drop=True)


def _find_log(raw, base, suffix):
    # Exact suffix first (e.g. _pure/_1k/_27k), then glob fallback so a casing
    # mismatch (_1K vs _1k) still resolves.
    p = raw / f"{base}_{suffix}.csv"
    if p.exists():
        return p
    cands = sorted(raw.glob(f"{base}_*.csv"))
    return cands[0] if cands else p

print("Helpers defined.")


## 3  Build one tier → `.inter`

In [ ]:
INTER_HEADER = ["user_id:token", "item_id:token", "timestamp:float"]
LOG_BASES = ["log_standard_4_08_to_4_21",
             "log_standard_4_22_to_5_08",
             "log_random_4_22_to_5_08"]


def build_tier(tier, write=True):
    # Build the RecBole .inter atomic file for one KuaiRand tier. Returns a stats dict.
    meta   = VERSIONS[tier]
    raw    = DATA_DIR / meta["variant"] / "data"
    paths  = [_find_log(raw, b, meta["suffix"]) for b in LOG_BASES]

    missing = [str(p) for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f"[{tier}] raw files not found (assume-pre-downloaded mode):\n  "
            + "\n  ".join(missing)
            + f"\nExpected under {raw.resolve()}"
        )

    t0 = time.perf_counter()
    df = pd.concat([load_and_filter(p) for p in paths], ignore_index=True)
    df = k_core_filter(df, MIN_USER_INTER, MIN_ITEM_INTER, K_CORE_ITERATE)
    df.sort_values(["user_id", "time_ms"], inplace=True)   # TO order per user
    df.reset_index(drop=True, inplace=True)

    stats = {"tier": tier, "dataset": meta["dataset"],
             "users": int(df["user_id"].nunique()),
             "items": int(df["video_id"].nunique()),
             "interactions": int(len(df)),
             "sec": round(time.perf_counter() - t0, 1)}
    stats["density_%"] = round(
        100 * stats["interactions"] / max(stats["users"] * stats["items"], 1), 5)

    if write and len(df):
        out_dir = OUTPUT_ROOT / meta["dataset"]
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{meta['dataset']}.inter"
        # RecBole atomic file: tab-separated, typed header, ordered by (user, time).
        df[["user_id", "video_id", "time_ms"]].to_csv(
            out_path, sep="\t", index=False, header=INTER_HEADER)
        stats["path"] = str(out_path)
        stats["size_mb"] = round(out_path.stat().st_size / 1e6, 1)

    return stats

print("build_tier() ready.")


## 4  Run the build

`light` + `medium` build by default. `heavy` (KuaiRand-27K, ~46 GB raw) is
**opt-in**: set `BUILD_HEAVY = True` once the 27K files are extracted and you're
on a machine with enough RAM. Any tier whose raw files are missing is skipped
with a clear message rather than crashing the run.

In [ ]:
BUILD_HEAVY = False
tiers = ["light", "medium"] + (["heavy"] if BUILD_HEAVY else [])

results = []
for tier in tiers:
    print(f"\n=== building {tier}  ({VERSIONS[tier]['variant']}) ===")
    try:
        s = build_tier(tier, write=True)
        results.append(s)
        print(f"  wrote {s.get('path', '<none>')}  ({s.get('size_mb', 0)} MB)")
        print(f"  users={s['users']:,}  items={s['items']:,}  "
              f"interactions={s['interactions']:,}  density={s['density_%']}%  ({s['sec']}s)")
    except FileNotFoundError as e:
        print(f"  SKIPPED — {e}")

if results:
    print("\n--- build summary ---")
    print(pd.DataFrame(results).set_index("tier").to_string())
else:
    print("\nNothing built. Place the raw KuaiRand CSVs under data/KuaiRand-*/data/ and re-run.")


## 5  Validate emitted atomic files

In [ ]:
# Re-read each .inter and assert it is a well-formed, leave-one-out-ready
# RecBole dataset: correct typed header, >= 3 interactions/user, time-sorted.
def validate(dataset):
    path = OUTPUT_ROOT / dataset / f"{dataset}.inter"
    if not path.exists():
        print(f"[{dataset}] no file to validate")
        return
    with path.open() as fh:
        header = fh.readline().rstrip("\n").split("\t")
    assert header == INTER_HEADER, f"{dataset}: bad header {header}"

    d = pd.read_csv(path, sep="\t")
    d.columns = ["user_id", "item_id", "timestamp"]
    per_user = d.groupby("user_id").size()
    assert per_user.min() >= 3, f"{dataset}: a user has < 3 interactions (LOO needs 3)"
    sorted_ok = bool(d.groupby("user_id")["timestamp"]
                      .apply(lambda s: s.is_monotonic_increasing).all())

    print(f"[{dataset}]  header OK | users={per_user.size:,} | rows={len(d):,} | "
          f"min/user={per_user.min()} | median/user={int(per_user.median())} | "
          f"sorted={sorted_ok}")
    print(d.head(3).to_string(index=False))

for s in results:
    validate(s["dataset"])


## 6  Train baselines on the built datasets

The atomic files match `configs/base.yaml` (`user_id` / `item_id` / `timestamp`,
`eval_args` = leave-one-out + `uni100`). Point the harness at the output root and
run — one `--dataset` per tier:

```bash
# One-time: set data_path in cafrec_harness/configs/base.yaml to the build output,
#   data_path: 'data/recbole'
# (the current value is a stale absolute path from another machine).

cd cafrec_harness

python run.py --models SASRec HGN --dataset kuairand_pure    # light
python run.py --models SASRec HGN --dataset kuairand_1k      # medium
python run.py --models SASRec HGN --dataset kuairand_27k     # heavy
```

Lock the resulting HR@10 / NDCG@10 / MRR into `PROJECT_LOG.md` — that closes
**RON-10 (HRNN/HGN)** and **RON-11 (SASRec)**, the last open Block-1 items
alongside the evaluation harness (RON-13/14/15).

**Note on the baseline swap:** the harness registry uses **HGN**, not the plan's
**HRNN**. If HRNN is required for the dissertation's H1 comparison, register it in
`cafrec/registry.py`; otherwise record the HGN substitution as a deviation in the
project log.